# EIA — MVP Validation

### Offline, neuromorphic, multi-signal field diagnostics

**What this notebook proves.** Compact models that (1) learn on **real clinical data**, (2) deploy on the committed neuromorphic chip (BrainChip Akida) with **almost no on-chip accuracy loss**, and (3) demonstrate the flagship differentiated capability — **detecting occult hemorrhage before vital signs move**.

Every performance number below is a **5-seed measurement** on the named public dataset. Sources and honest caveats are in the last two cells. This notebook is self-contained — it needs only `numpy` and `matplotlib` (already in Colab); no repo clone or install required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':120,'font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})
FLOAT_C='#2E5A88'   # float model
CHIP_C='#3BA99C'    # on-chip (Akida)
XYLO_C='#C0561F'    # prior path (Xylo)

## The problem — and where the device is aimed

Most preventable prehospital deaths come from a very short list, dominated by hemorrhage. The device's priorities follow that lethality: the flagship signal targets **occult hemorrhage** — the largest share, and the one invisible to a manual pulse — with airway and tension pneumothorax on the roadmap.

*Source: battlefield preventable-death data (see the brief's Sources).*

In [ ]:
from matplotlib.patches import Patch
causes=['Hemorrhage','Airway\nobstruction','Tension\npneumothorax']; pct=[91,7.9,1.1]
colors=['#C0561F','#C99A2E','#C99A2E']; y=[2,1,0]
fig,ax=plt.subplots(figsize=(8,3.1))
ax.barh(y,pct,color=colors,height=0.55)
ax.set_yticks(y); ax.set_yticklabels(causes)
ax.set_xlim(0,100); ax.set_xlabel('share of preventable prehospital deaths (%)')
ax.set_title('Preventable field deaths concentrate in a short list',fontweight='bold')
for yi,p in zip(y,pct):
    ax.text(p+1.5,yi,(f'{p:.0f}%' if p>=10 else f'{p:.1f}%'),va='center',fontweight='bold')
ax.legend([Patch(color='#C0561F'),Patch(color='#C99A2E')],
          ['flagship — occult CRM (demonstrated)','roadmap'],frameon=False,loc='lower right',fontsize=9)
plt.tight_layout(); plt.show()

## 1 — It learns on real clinical data

Three independent modalities, each trained and evaluated on a separate public clinical dataset. All clear the ~0.9 AUROC range that the literature benchmarks for these tasks — genuine detection, not chance.

In [ ]:
labels=['ECG\narrhythmia\n(MIT-BIH)','Heart sounds\n(CinC 2016)','ECG — MI\n(PTB-XL)']
auroc=[0.970,0.945,0.890]
fig,ax=plt.subplots(figsize=(7,3.8))
bars=ax.bar(labels,auroc,color=FLOAT_C,width=0.55)
ax.axhline(0.5,ls='--',c='grey',lw=1); ax.text(2.3,0.515,'chance',color='grey',fontsize=9)
ax.set_ylim(0.5,1.0); ax.set_ylabel('AUROC — float model (5 seeds)')
ax.set_title('Learns on real clinical data',fontweight='bold')
for b,v in zip(bars,auroc):
    ax.text(b.get_x()+b.get_width()/2,v+0.006,f'{v:.3f}',ha='center',fontweight='bold')
plt.tight_layout(); plt.show()

## 2 — It deploys on the committed chip with essentially no accuracy loss

The whole thesis is *offline, low-power inference*. Here each model is quantized and run on BrainChip's Akida simulator. The on-chip result is statistically indistinguishable from the full-precision float model.

In [ ]:
labels=['ECG arrhythmia','Heart sounds','ECG — MI']
fl=[0.928,0.865,0.795]; ak=[0.926,0.867,0.806]
x=np.arange(3); w=0.36
fig,ax=plt.subplots(figsize=(7,3.8))
ax.bar(x-w/2,fl,w,label='Float model',color=FLOAT_C)
ax.bar(x+w/2,ak,w,label='On-chip (Akida simulator)',color=CHIP_C)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0.5,1.0)
ax.set_ylabel('Balanced accuracy'); ax.legend(frameon=False)
ax.set_title('Deploys on the committed chip with ~no accuracy loss',fontweight='bold')
plt.tight_layout(); plt.show()

## 3 — Why the chip choice is validated, not assumed

Float↔on-chip **agreement** = the fraction of cases where the deployed model matches the float model. On the earlier spiking-network path (SynSense Xylo) this dropped sharply; on the committed Akida path it stays near 1.0 across every modality. This is the evidence behind the single-vendor Akida decision.

In [ ]:
labels=['ECG arrhythmia','Heart sounds','ECG — MI']
xylo=[0.56,0.779,0]; akida=[0.981,0.954,0.917]
x=np.arange(3); w=0.36
fig,ax=plt.subplots(figsize=(7,3.8))
ax.bar(x-w/2,xylo,w,label='Prior path (Xylo, spiking net)',color=XYLO_C)
ax.bar(x+w/2,akida,w,label='Committed path (Akida, quantized CNN)',color=CHIP_C)
ax.text(x[2]-w/2,0.03,'n/a',ha='center',color='grey',fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0,1.05)
ax.set_ylabel('Float↔on-chip agreement'); ax.legend(frameon=False,loc='lower left')
ax.set_title('The on-chip fidelity gap closes on Akida',fontweight='bold')
plt.tight_layout(); plt.show()

## 4 — The flagship: detecting hemorrhage *before* vital signs move

The differentiated capability — the one thing a responder taking a pulse cannot do. As simulated blood volume falls, the PPG pulse loses amplitude and its dicrotic notch blunts **while heart rate is still normal**. The device flags the compromise in that *occult band* (shaded), before compensation fails.

*Illustrative of the synthetic time-resolved CRM generator (measured: float AUROC 0.975; docs/synthetic_crm_results.md). Synthetic — proves the capability and pipeline, not clinical accuracy; real induced-hypovolemia (LBNP) data is being requested.*

In [ ]:
def pulse(n,r):
    t=np.linspace(0,1,n); amp=0.35+0.65*r
    notch=max(0.0,(r-0.15))/0.85; width=0.052-0.016*(1-r)
    p=(amp*np.exp(-((t-0.30)**2)/(2*width**2))
       +amp*0.16*notch*np.exp(-((t-0.55)**2)/(2*0.028**2))
       +amp*0.26*notch*np.exp(-((t-0.63)**2)/(2*0.07**2)))
    return t,p

fig,axes=plt.subplots(1,2,figsize=(10,3.9))
for r,c,lab in [(1.0,FLOAT_C,'full reserve'),(0.6,'#C99A2E','r=0.6 (occult)'),(0.3,XYLO_C,'r=0.3 (crashing)')]:
    t,p=pulse(200,r); axes[0].plot(t,p,color=c,lw=2,label=lab)
axes[0].set_title('PPG pulse morphology falls with reserve',fontweight='bold')
axes[0].set_xlabel('one pulse'); axes[0].set_yticks([]); axes[0].legend(frameon=False,fontsize=9)

r=np.linspace(1,0,200); prog=100*(1-r)
hr=np.where(r>=0.30,80.0,80.0+(0.30-r)/0.30*50)
ax2=axes[1]; l1,=ax2.plot(prog,r,color=FLOAT_C,lw=2)
ax2b=ax2.twinx(); l2,=ax2b.plot(prog,hr,color=XYLO_C,lw=2)
ax2.axvspan(50,70,color='#F2C94C',alpha=0.30)
ax2.text(60,0.92,'flagged compromised,\nHR still normal',ha='center',fontsize=8.5,color='#8a6d1a')
ax2.set_title('Detected before vitals move (occult band)',fontweight='bold')
ax2.set_xlabel('progression of blood loss →')
ax2.set_ylabel('reserve (0–1)',color=FLOAT_C); ax2b.set_ylabel('heart rate (bpm)',color=XYLO_C)
ax2.set_ylim(0,1.05)
ax2.legend([l1,l2],['reserve','heart rate'],frameon=False,fontsize=9,loc='lower left')
plt.tight_layout(); plt.show()

## MVP scorecard

| Modality | Dataset | Real data | Learns (float AUROC) | Deploys on Akida (agreement) | Role |
|---|---|:---:|:---:|:---:|---|
| ECG — arrhythmia | MIT-BIH | ✅ | 0.970 | 0.981 | Circulation |
| Heart sounds | PhysioNet CinC 2016 | ✅ | 0.945 | 0.954 | Auscultation |
| ECG — myocardial infarction | PTB-XL | ✅ | 0.890 | 0.917 | Circulation — **differentiated (heart attack)** |
| Hemorrhage / CRM | synthetic | ⚠️ synthetic | 0.975 | 0.818 | **Flagship (occult)** — real data pending |

**Three real-data clinical modalities that learn and deploy faithfully on the committed neuromorphic chip, plus the flagship occult-hemorrhage capability demonstrated in physiologically-grounded simulation.**

## Traction across MARCH

Coverage of the MARCH framework by maturity. **Circulation** is validated on real data and deploys on-chip (three modalities); **Massive hemorrhage** is demonstrated in simulation with real data requested; **Respiration** is spec'd; **Head** was attempted (EEG) and paused; **Airway** is future.

**The next differentiated component is the multi-signal fusion layer** — combining these instrumented signals into diagnoses no single sensor or physical exam can produce (shock etiology, tension pneumothorax).

In [ ]:
march=[('C — Circulation',4.0,'real + on-chip'),
       ('M — Massive hemorrhage',2.0,'demonstrated (sim)'),
       ('R — Respiration',1.0,"spec'd"),
       ('H — Head',0.7,'attempted, paused'),
       ('A — Airway',0.3,'future')]
stages=['not\nstarted','planned','demo\n(sim)','validated\n(real)','on-chip']
def col(v): return '#3BA99C' if v>=3 else ('#C99A2E' if v>=2 else ('#7f9bb3' if v>=1 else '#cccccc'))
y=np.arange(len(march))[::-1]
fig,ax=plt.subplots(figsize=(9,3.5))
for yi,(name,v,lab) in zip(y,march):
    ax.barh(yi,max(v,0.06),color=col(v),height=0.55)
    ax.text(min(v+0.08,3.95),yi,lab,va='center',ha='left',fontsize=8.5,color='#333')
ax.set_yticks(y); ax.set_yticklabels([m[0] for m in march])
ax.set_xlim(0,4); ax.set_xticks(range(5)); ax.set_xticklabels(stages,fontsize=8.3)
ax.set_xlabel('maturity'); ax.set_title('MARCH coverage & traction',fontweight='bold')
plt.tight_layout(); plt.show()

## Hardware / system architecture

Akida is a **digital compute core with no analog front-end** — no sensor connects to it directly. A host MCU/SBC acquires every sensor, runs the same preprocessing the software above already does (windowing, band-pass, delta/band-power encoding, per-window normalization), feeds Akida the quantized tensor, and reads back the inference.

*Akida is a digital core with no analog front-end — every sensor is digital-output and bridged by the host; the working modalities reduce to three sensor front-ends plus capnography. (See `docs/hardware_bom.md`.)*

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.3))
ax.set_xlim(0, 10.4); ax.set_ylim(-0.3, 5.8); ax.axis('off')

def box(x, y, w, h, label, color, fontsize=9.5):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='black', lw=1.1, zorder=2))
    ax.text(x+w/2, y+h/2, label, ha='center', va='center', fontsize=fontsize,
            color='white', fontweight='bold', zorder=3)

sensors = [
    ('ECG AFE\nSPI', 4.55),
    ('PPG\nI2C', 3.15),
    ('MEMS mic\nI2S', 1.75),
    ('Capnography (NDIR)\nUART', 0.35),
]
sw, sh = 2.5, 1.05
hx, hy, hw, hh = 3.6, 0.1, 2.9, 5.15
for label, y in sensors:
    box(0.15, y, sw, sh, label, '#7f9bb3', fontsize=8.7)
    ax.annotate('', xy=(hx, hy+hh/2), xytext=(0.15+sw, y+sh/2),
                arrowprops=dict(arrowstyle='-', color='#888', lw=1.3,
                                 connectionstyle='arc3,rad=0.0'), zorder=1)

box(hx, hy, hw, hh, 'Host MCU / SBC\n\nacquire · window ·\npreprocess/encode ·\nfeed Akida · fuse', FLOAT_C, fontsize=9.5)

ax.annotate('', xy=(hx+hw+1.0, hy+hh/2), xytext=(hx+hw, hy+hh/2),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

box(hx+hw+1.0, hy+0.9, 2.3, hh-1.8, 'Akida\n\nquantized-CNN\ninference', CHIP_C, fontsize=9.5)

ax.set_title('Sensor front-ends bridge to Akida through a host MCU/SBC', fontweight='bold')
plt.tight_layout(); plt.show()

## Current stage & path to device

Three-stage path from validated software to a fielded device. **The board prototype proves the SYSTEM** — sensors → inference → fusion, working end to end. **The cloud/Pico track proves the POWER** — sub-milliwatt always-on inference. These are two distinct claims, demonstrated on two different platforms — not the same result.

In [ ]:
stages = [
    ('Software +\nAkida-simulator\nvalidation', 'DONE', 'three real modalities', CHIP_C, 'white'),
    ('Sensor-integrated prototype\n(off-the-shelf Akida board)', 'IN PROGRESS',
     'bring-up — no custom silicon\nor PCB on the critical path', '#C99A2E', 'white'),
    ('Sub-milliwatt field device\n(Akida Pico)', 'NEXT',
     'power validated on\nthe cloud FPGA', '#cccccc', '#333333'),
]
fig, ax = plt.subplots(figsize=(10, 3.4))
x = np.arange(3)
for xi, (title, status, note, color, txt_c) in zip(x, stages):
    ax.add_patch(plt.Rectangle((xi-0.42, 0.25), 0.84, 0.75, facecolor=color,
                                edgecolor='black', lw=1.1, zorder=2))
    ax.text(xi, 0.625, status, ha='center', va='center', fontsize=10.5,
            fontweight='bold', color=txt_c, zorder=3)
    ax.text(xi, -0.05, title, ha='center', va='top', fontsize=9.3, fontweight='bold')
    ax.text(xi, -0.62, note, ha='center', va='top', fontsize=8.2, color='#555')
    if xi < 2:
        ax.annotate('', xy=(xi+0.58, 0.625), xytext=(xi+0.42, 0.625),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.6))
ax.set_xlim(-0.75, 2.75); ax.set_ylim(-1.15, 1.1); ax.axis('off')
ax.set_title('Path to device — three stages', fontweight='bold')
plt.tight_layout(); plt.show()

## Honest caveats (stated up front)

- **On-chip = BrainChip's *software* simulator.** BrainChip does not publish a bit/cycle-exact-to-silicon claim for it, so "deploys faithfully" is measured in simulation; real hardware may differ. Measured on the **Akida 2.0 FPGA** target, not the sub-milliwatt **Akida Pico** committed for the always-on cluster.
- **Models are quantized CNNs, not spiking networks.** The on-chip fidelity is strong partly because a feed-forward CNN avoids the temporal-state accumulation that hurt the earlier spiking path — a chip-**and**-architecture result, not chip alone.
- **CRM / hemorrhage is synthetic**, separable by construction. It proves the pipeline and that a time-aligned label is learnable — **not** clinical hemorrhage-detection accuracy. Real induced-hypovolemia (LBNP) data has been requested.
- **Energy is not yet measured on the Akida-CNN path.** The milliwatt claim currently rests on BrainChip's published always-on specs and an analytical model; a measured energy number is pending.
- **The board prototype uses a ~1 W-class AKD1000/AKD1500 dev board, not the sub-mW Pico** — it proves function, not the always-on power claim (that's the cloud/Pico track).
- **These are single-signal detectors.** The multi-signal diagnostic fusion layer (shock etiology, tension pneumothorax) — where much of the device's value lives — is the next component, not yet built.

## Provenance / reproduce

- **Repo:** `github.com/dcaglar-28/NeuroSoC`
- **Measured with:** `scripts/akida_docker_run.sh python scripts/akida_verify.py --modality {ecg,heart,mi} --real --n-seeds 5`
- **Full write-ups:** `docs/akida_ecg_results.md`, `docs/akida_heart_results.md`, `docs/ptbxl_mi_results.md`, `docs/synthetic_crm_results.md`
- **Datasets:** MIT-BIH Arrhythmia, PhysioNet/CinC 2016, PTB-XL — all public (PhysioNet).